# 3. 캐시 패턴

가장 많이 쓰는 캐싱 방식이다.
  캐시에 있으면 그대로 쓰고, 없으면 DB 에서 읽어와 캐시에 넣어둔다.

위에서부터 셀을 하나씩 실행합니다 (`Shift + Enter`).

TODO 를 채우기 전에는 결과가 비어 있게 나온다 (None, [], {}).
`...` 은 파이썬 문법상 유효해서 오류 없이 지나가기 때문이다.
결과가 비어 있으면 고장난 것이 아니라 아직 안 채운 것이다.

In [1]:
import json
import time

from db import supabase
from redis_client import r

In [5]:
def get_profile(profile_id):
    """프로필을 가져온다. 캐시에 있으면 캐시에서, 없으면 DB 에서."""
    key = "day14:profile:" + profile_id

    # TODO 1. 캐시에 있으면 "캐시에서 가져옴" 을 출력하고 그 값을 돌려준다.
    #         -> 캐시에 있으면 1 없으면 0
    #         힌트: r.get(key) 에 값이 있으면 json.loads 로 되돌린다

    # 캐시에서 값 가져옴 / z캐시에서 검색
    cached = r.get(key) 

    #miss 확인
    if cached:
        print("  캐시에서 가져옴")
        return json.loads(cached) #있으면 json으로 바꿔서 돌려주기

    # TODO 2. 없으면 "DB 에서 가져옴" 을 출력하고 supabase 에서 읽어온다.
    #         읽어온 값을 5초 만료로 캐시에 넣고 돌려준다.
    #         힌트: supabase.table("profiles").select("*").eq("id", profile_id).execute().data
    #               r.set(key, json.dumps(값, default=str), ex=5)
    
    # miss인 경우 supabase 조회
    print("  DB 에서 가져옴")
    rows = supabase.table("profiles").select("*").eq("id", profile_id).execute().data
    print(f"조회 결과 : {rows}")

    #조회 결과가 없으면
    if not rows:
        return None

    #캐시에 넣기
    profile = rows[0]
    # ensure_ascii=False : 한글 깨짐 방지
    r.set(key, json.dumps(profile, default=str, ensure_ascii=False), ex=5)
    return profile

## 0. 대상 준비

In [3]:
rows = supabase.table("profiles").select("id, username").limit(1).execute().data
if not rows:
    print("profiles 가 비어 있다. 앞 차수 실습을 먼저 한다.")
    sys.exit()

profile_id = rows[0]["id"]
print("대상:", rows[0]["username"])

r.delete("day14:profile:" + profile_id)

대상: bbb


0

## 1. 캐시가 없을 때와 있을 때

In [6]:
print("1번째 호출")
start = time.time()
get_profile(profile_id)
print("  걸린 시간:", round(time.time() - start, 3), "초")

1번째 호출
  DB 에서 가져옴
조회 결과 : [{'id': '353d166f-e9fc-4016-bc61-12ab0ee8197e', 'username': 'bbb', 'created_at': '2026-08-25T08:30:33.742881+00:00'}]
  걸린 시간: 0.656 초


In [7]:

print("2번째 호출")
start = time.time()
get_profile(profile_id)
print("  걸린 시간:", round(time.time() - start, 3), "초")

2번째 호출
  캐시에서 가져옴
  걸린 시간: 0.007 초


In [8]:
print("3번째 호출")
start = time.time()
get_profile(profile_id)
print("  걸린 시간:", round(time.time() - start, 3), "초")

3번째 호출
  캐시에서 가져옴
  걸린 시간: 0.009 초


## 2. 5초가 지나면

In [9]:
print("5초 기다린다...")
time.sleep(6)

print("4번째 호출")
start = time.time()
get_profile(profile_id)
print("  걸린 시간:", round(time.time() - start, 3), "초")

print()
print("캐시가 사라져서 다시 DB 로 갔다.")

5초 기다린다...
4번째 호출
  DB 에서 가져옴
조회 결과 : [{'id': '353d166f-e9fc-4016-bc61-12ab0ee8197e', 'username': 'bbb', 'created_at': '2026-08-25T08:30:33.742881+00:00'}]
  걸린 시간: 0.39 초

캐시가 사라져서 다시 DB 로 갔다.


## 3. 값이 바뀌면 캐시를 지워야 한다

In [12]:
key = "day14:profile:" + profile_id

# 캐시에 옛날 값이 들어 있는 상황을 만든다.
r.set(key, json.dumps({"username": "옛날이름"}), ex=300)
print("캐시에 옛날 값을 넣어뒀다.")

profile = get_profile(profile_id)
if profile:
    print("  가져온 이름:", profile["username"], "  <- 틀린 값이다")

캐시에 옛날 값을 넣어뒀다.
  캐시에서 가져옴
  가져온 이름: 옛날이름   <- 틀린 값이다


In [13]:


print()
print("캐시를 지운다.")
# TODO 3. 캐시를 지운다.  힌트: r.delete(key)
r.delete(key)


캐시를 지운다.


1

In [14]:
profile = get_profile(profile_id)
if profile:
    print("  가져온 이름:", profile["username"], "  <- 올바른 값이다")

print()
print("값을 바꾸는 코드가 캐시도 함께 지워야 한다.")
print("이 한 줄을 빠뜨리면 '방금 저장했는데 안 보인다'는 문제가 생긴다.")

  DB 에서 가져옴
조회 결과 : [{'id': '353d166f-e9fc-4016-bc61-12ab0ee8197e', 'username': 'bbb', 'created_at': '2026-08-25T08:30:33.742881+00:00'}]
  가져온 이름: bbb   <- 올바른 값이다

값을 바꾸는 코드가 캐시도 함께 지워야 한다.
이 한 줄을 빠뜨리면 '방금 저장했는데 안 보인다'는 문제가 생긴다.


## 4. 정리

In [15]:
keys = r.keys("day14:*")
for k in keys:
    r.delete(k)

print(len(keys), "개 삭제")
print("남은 키 개수:", r.dbsize())

1 개 삭제
남은 키 개수: 0
